## Exploratory Data Analysis
Week 2 submission

Contents:

- UPDATE

In [9]:
#import relevant tools
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold



In [10]:
#get data
data = pd.read_csv("data/core/systems_cleaned.csv")
#focus only on the locations that have irradiance data
#data_irrad = data.loc[data["has_irrad_data"]]
#data_irrad
data
#note that only 5 of these are from the prize_data, so they might be the only once with enough irradiance data?

,system_id,system_public_name,site_location,timezone_or_utc_offset,latitude,longitude,elevation_m,dc_capacity_kW,kg_climate,pvcz_composite,...,has_ambient_temperature_data,has_temperature_data,has_power_data,has_current_data,has_voltage_data,has_ac_data,has_dc_data,module_type,simplified_type,system_source
0,2,Residential 1a,"Lakewood, CO",America/Denver,39.72140,-105.09720,1675.000000,2.912,Dfb,12,...,False,True,True,True,True,True,True,multi-Si,multicrystalline_Si,PVDAQ General
1,3,Residential 1b,"Lakewood, CO",America/Denver,39.72140,-105.09720,1675.000000,2.720,Dfb,12,...,False,True,True,True,True,True,True,amorphous si,thin_film,PVDAQ General
2,4,NREL x-Si -1,"Golden, CO",7,39.74060,-105.17740,1795.300000,1.000,BSk,12,...,True,True,True,True,True,True,True,mono-Si,monocrystalline_Si,PVDAQ General
3,10,NREL CIS -1,"Golden, CO",7,39.74040,-105.17740,1792.800000,1.120,BSk,12,...,True,True,True,True,True,True,True,cis family thin-film,thin_film,PVDAQ General
4,33,Silicor Materials,"Golden, CO",7,39.74040,-105.17720,1794.000000,2.400,BSk,12,...,True,True,True,True,True,True,True,Unknown,unknown,PVDAQ General
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1854,14760,PVDB_TADHC1036224,NaN,America/Los_Angeles,33.71677,-117.90680,16.204149,NaN,NaN,33,...,False,True,True,True,True,True,True,SJT,heterojunction,PVDB
1855,14761,PVDB_TADHC1036570,NaN,America/Los_Angeles,33.77269,-117.80076,83.785385,NaN,NaN,34,...,False,True,True,True,True,True,True,SJT,heterojunction,PVDB
1856,14762,PVDB_TADKC1017785,NaN,America/Los_Angeles,33.75678,-117.84557,49.351692,NaN,NaN,34,...,False,True,True,True,True,True,True,SJT,heterojunction,PVDB
1857,14763,PVDB_TADKC1018154,NaN,America/Los_Angeles,33.73392,-117.89290,20.376822,NaN,NaN,34,...,False,True,True,True,True,True,True,SJT,heterojunction,PVDB


In [15]:
data.columns

data_prize = data.loc[data['is_prize_data']]
data_prize = data_prize.drop(columns = ['dataset_size_mb', 'available_sensor_channels',
       'qa_status', 'qa_issue', 'first_year', 'is_prize_data',
       'is_lake_parquet_data', 'is_lake_csv_data', 'pvcz_t_rack',
       'pvcz_t_roof', 'pvcz_humidity', 'pvcz_wind', 'tracking', 'type',
       'azimuth', 'tilt'])
data_prize

,system_id,system_public_name,site_location,timezone_or_utc_offset,latitude,longitude,elevation_m,dc_capacity_kW,kg_climate,pvcz_composite,...,has_ambient_temperature_data,has_temperature_data,has_power_data,has_current_data,has_voltage_data,has_ac_data,has_dc_data,module_type,simplified_type,system_source
154,2105,Maui Ocean Center,"Maui, HI",10,20.792530,-156.512000,10.0,110.0,Cfb,36,...,True,True,True,False,True,True,True,mono-Si,monocrystalline_Si,Prize
155,2107,Farm Solar Array,"Arbuckle, CA",PST8PDT,38.996306,-122.134111,10.0,893.0,Csa,33,...,True,True,True,True,True,True,True,mono-Si,monocrystalline_Si,Prize
160,7333,Shine On Solar Facility - 10sec,"Levis, CA",8,36.578000,-120.401600,330.1,205300.0,Bsk,34,...,True,True,True,True,True,True,True,multi-Si,multicrystalline_Si,Prize
161,9068,SR_CO,"Kersey, CO",7,40.386400,-104.551200,1407.0,4738.0,Bsk,12,...,True,True,True,True,True,True,True,CdTe,thin_film,Prize
162,9069,Simon Solar Farm,"Social Circle, GA",5,33.676200,-83.676000,330.1,33000.0,Cfa,35,...,True,True,True,True,True,True,True,multi-Si,multicrystalline_Si,Prize


In [17]:
data_prize.columns
data_prize = data_prize.drop(columns = ['has_irradiance_data',
       'has_ambient_temperature_data', 'has_temperature_data',
       'has_power_data', 'has_current_data', 'has_voltage_data', 'has_ac_data',
       'has_dc_data', 'module_type', 'simplified_type', 'system_source'])
data_prize

,system_id,system_public_name,site_location,timezone_or_utc_offset,latitude,longitude,elevation_m,dc_capacity_kW,kg_climate,pvcz_composite,first_timestamp,last_timestamp,years,number_records
154,2105,Maui Ocean Center,"Maui, HI",10,20.792530,-156.512000,10.0,110.0,Cfb,36,2018-12-15 00:15:00,2023-11-15 12:21:00,4.9205,1.581011e+07
155,2107,Farm Solar Array,"Arbuckle, CA",PST8PDT,38.996306,-122.134111,10.0,893.0,Csa,33,2017-01-01 00:00:00,2024-11-30 23:45:00,7.9178,8.936954e+07
160,7333,Shine On Solar Facility - 10sec,"Levis, CA",8,36.578000,-120.401600,330.1,205300.0,Bsk,34,2016-10-31 21:00:00,2023-10-31 15:59:00,7.0027,4.064245e+10
161,9068,SR_CO,"Kersey, CO",7,40.386400,-104.551200,1407.0,4738.0,Bsk,12,2017-08-23 17:40:00,2025-04-30 14:20:00,7.6904,3.296997e+08
162,9069,Simon Solar Farm,"Social Circle, GA",5,33.676200,-83.676000,330.1,33000.0,Cfa,35,2016-02-17 18:05:00,2023-11-29 14:50:00,7.7863,1.758480e+09


In [21]:
list(data_prize.columns)

['system_id',
 'system_public_name',
 'site_location',
 'timezone_or_utc_offset',
 'latitude',
 'longitude',
 'elevation_m',
 'dc_capacity_kW',
 'kg_climate',
 'pvcz_composite',
 'first_timestamp',
 'last_timestamp',
 'years',
 'number_records']

In [7]:
#figure out where all these things are located
import plotly.express as px
fig = px.density_map(data_irrad, 
                     lat='latitude', 
                     lon='longitude', 
                     z='elevation_m', #ideally irradiance or degradation
                     radius=20,
                    center=dict(lat=40, lon=-98), 
                    zoom=2.8,
                    map_style="open-street-map")
fig.show()

NameError: name 'data_irrad' is not defined

In [ ]:
#plot variables against each other
#first, figure out what the variables even are
data_irrad.columns

Index(['system_id', 'system_public_name', 'site_location',
       'timezone_or_utc_offset', 'latitude', 'longitude', 'elevation_m',
       'dc_capacity_kW', 'kg_climate', 'pvcz_composite', 'pvcz_t_rack',
       'pvcz_t_roof', 'pvcz_humidity', 'pvcz_wind', 'tracking', 'type',
       'azimuth', 'tilt', 'first_timestamp', 'last_timestamp', 'years',
       'number_records', 'dataset_size_mb', 'available_sensor_channels',
       'qa_status', 'qa_issue', 'first_year', 'is_prize_data',
       'is_lake_parquet_data', 'is_lake_csv_data', 'has_irrad_data'],
      dtype='str')

In [ ]:
#what are the types in each column?
data_irrad = data.loc[data["has_irrad_data"]]
data_irrad=data_irrad.drop(columns=["is_prize_data",
                                    "is_lake_parquet_data",
                                    "is_lake_csv_data",
                                    "has_irrad_data",
                                    "timezone_or_utc_offset",
                                    "first_timestamp",
                                    "last_timestamp",
                                    "system_public_name",
                                    "first_year"])
data_irrad.info()

<class 'pandas.DataFrame'>
Index: 66 entries, 0 to 163
Data columns (total 22 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   system_id                  66 non-null     int64  
 1   site_location              66 non-null     str    
 2   latitude                   66 non-null     float64
 3   longitude                  66 non-null     float64
 4   elevation_m                66 non-null     float64
 5   dc_capacity_kW             47 non-null     float64
 6   kg_climate                 66 non-null     str    
 7   pvcz_composite             66 non-null     int64  
 8   pvcz_t_rack                66 non-null     int64  
 9   pvcz_t_roof                66 non-null     int64  
 10  pvcz_humidity              66 non-null     int64  
 11  pvcz_wind                  66 non-null     int64  
 12  tracking                   65 non-null     str    
 13  type                       65 non-null     str    
 14  azimuth    